# Task 09 -- Growing DMD modes (`docs/open_questions.md` §5.5)

Surveys the distribution of `|lambda_k|` across ranks/windows to
distinguish genuine growth from noise-attributable growth, per
`dynamic_mode_decomposition.md` §6's proposed clip-and-flag mitigation.
See `tasks/09-growing-dmd-modes.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2026, 3, 1, tzinfo=timezone.utc)

VARIABLE = "wsh"
X_raw = load_training_window(DATA_DIR, T_START, T_END, [VARIABLE])
X = X_raw[VARIABLE].transpose("wave_lat", "wave_lon", "time").values

ocean_mask = ~np.isnan(X).any(axis=-1)
tucker_mask = np.broadcast_to(ocean_mask[:, :, None], X.shape).astype(np.float64)
X_filled = np.where(ocean_mask[:, :, None], X, 0.0)

core, _ = partial_tucker(
    X_filled, modes=[0, 1, 2], rank=[10, 20, X_filled.shape[-1]],
    init="random", tol=1e-5, n_iter_max=2000, random_state=SEED, mask=tucker_mask,
)


## Sweep `|lambda_k|` across DMD rank and across sliding sub-windows

In [ ]:
from train.dmd import dmd_fit

dt = 3.0 * 3600.0
T = core.shape[0]

print("--- across r_dmd, full window ---")
for r_dmd in [5, 10, 20, 40]:
    dmd = dmd_fit(core, r_dmd=r_dmd, dt=dt, rng=rng)
    growing = np.abs(dmd.Lambda) > 1.0 + 1e-6
    print(f"r_dmd={r_dmd}: {growing.sum()} growing modes, "
          f"max|lambda|={np.abs(dmd.Lambda).max():.4f}")

print("\n--- fixed r_dmd=20, sliding sub-windows (does growth persist or wander?) ---")
window_len = T // 3
for start in range(0, T - window_len, window_len // 2):
    sub_core = core[start : start + window_len]
    dmd = dmd_fit(sub_core, r_dmd=20, dt=dt, rng=rng)
    growing = np.abs(dmd.Lambda) > 1.0 + 1e-6
    print(f"window [{start}:{start + window_len}]: {growing.sum()} growing modes, "
          f"max|lambda|={np.abs(dmd.Lambda).max():.4f}")


## Decision-rule diagnostic

A mode that is growing consistently across sub-windows and ranks is a
stronger candidate for "genuine" growth (e.g. a real, sustained storm
intensification within the window) than one whose growth appears/disappears
with rank or window choice (noise-attributable). Use this to calibrate the
"small margin: clip" vs. "large margin: flag for review" thresholds
`dynamic_mode_decomposition.md` §6 proposes. Record in
`tasks/09-growing-dmd-modes.md` §4.